# 04. 하이퍼파라미터 세밀 탐색

파생변수 19개(기존 17개 + `anticipatory_stress` + `cardio_metabolic_load`, 09/10 CV 검증 완료) +
#00과 동일한 인코딩을 기반으로, #03(`regularized`)에서 발견한 방향을 더 세밀하게 탐색합니다.

비교 기준: #00 (기본) 0.2117 / #03 (regularized) 0.1855

In [1]:
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

RANDOM_STATE = 42  # 그라운드룰 1: 항상 42로 고정

## 1. Data Load

In [2]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

print('train:', train.shape, '/ test:', test.shape)

train: (3000, 18) / test: (3000, 17)


## 2. 결측치 및 중복행 처리 (0909 ver, 팀 확정본)

In [3]:
# 중복 행 제거 (ID 제외 기준) - train에만 적용, test는 제거하지 않음
train = train.drop_duplicates(
    subset=[col for col in train.columns if col not in ['ID']]
).reset_index(drop=True)

# 근로시간 결측치: 0 처리
train['mean_working'] = train['mean_working'].fillna(0)
test['mean_working'] = test['mean_working'].fillna(0)

# 범주형 결측치: 독립 범주 신설
for col in ['medical_history', 'family_medical_history']:
    train[col] = train[col].fillna('None')
    test[col] = test[col].fillna('None')

train['edu_level'] = train['edu_level'].fillna('Unknown')
test['edu_level'] = test['edu_level'].fillna('Unknown')

print('결측치 처리 후 남은 결측 개수 - train:', train.isnull().sum().sum(), '/ test:', test.isnull().sum().sum())
print('중복 제거 후 train shape:', train.shape)

결측치 처리 후 남은 결측 개수 - train: 0 / test: 0
중복 제거 후 train shape: (2994, 18)


## 3. 파생변수 생성 (19개 확정본)

In [4]:
def add_features(df):
    data = df.copy()
    has_disease = (data['medical_history'] != 'None').astype(int)

    # 1. 과로 및 생활 리듬
    data['is_overworking'] = (data['mean_working'] >= 10).astype(int)
    data['work_sleep_risk'] = ((data['mean_working'] >= 9) & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)
    data['oversleep_low_activity'] = ((data['sleep_pattern'] == 'oversleeping') & (data['activity'] == 'light')).astype(int)
    data['working_age_ratio'] = data['mean_working'] / (data['age'] + 1)
    data['activity_sleep_mismatch'] = ((data['activity'] == 'intense') & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)

    # 2. 질환 및 유전력
    data['smoker_with_disease'] = ((data['smoke_status'] == 'current-smoker') & (has_disease == 1)).astype(int)
    data['age_disease_interaction'] = data['age'] * has_disease
    data['has_medical_history'] = has_disease
    data['has_family_history'] = (data['family_medical_history'] != 'None').astype(int)
    data['total_disease_burden'] = data['has_medical_history'] + data['has_family_history']
    data['genetic_risk_match'] = ((data['medical_history'] == data['family_medical_history']) & (has_disease == 1)).astype(int)

    # 3. 심혈관 및 신체
    data['bmi'] = data['weight'] / ((data['height'] / 100) ** 2)
    data['pulse_pressure'] = data['systolic_blood_pressure'] - data['diastolic_blood_pressure']
    data['map'] = data['diastolic_blood_pressure'] + (data['pulse_pressure'] / 3)
    data['is_hypertension'] = ((data['systolic_blood_pressure'] >= 140) | (data['diastolic_blood_pressure'] >= 90)).astype(int)

    # 4. 대사 및 노화
    data['is_low_bone_density'] = (data['bone_density'] < 0).astype(int)
    data['glucose_chol_ratio'] = data['glucose'] / (data['cholesterol'] + 1)

    # 5. 추가 검증된 2개 (09/10 CV 테스트 결과 채택)
    data['anticipatory_stress'] = ((data['family_medical_history'] != 'None') & (data['medical_history'] == 'None')).astype(int)
    data['cardio_metabolic_load'] = data['map'] * data['bmi']

    return data

train = add_features(train)
test = add_features(test)

print('파생변수 추가 후 train shape:', train.shape, '(19개 확정)')

파생변수 추가 후 train shape: (2994, 37) (19개 확정)


## 4. 인코딩 (#00과 동일: Ordinal + LabelEncoder 혼합)

In [5]:
activity_map = {'light': 0, 'moderate': 1, 'intense': 2}
edu_map = {'Unknown': 0, 'high school diploma': 1, 'bachelors degree': 2, 'graduate degree': 3}

train['activity'] = train['activity'].map(activity_map)
test['activity'] = test['activity'].map(activity_map)
train['edu_level'] = train['edu_level'].map(edu_map)
test['edu_level'] = test['edu_level'].map(edu_map)

nominal_cols = ['gender', 'smoke_status', 'medical_history', 'family_medical_history', 'sleep_pattern']

for feature in nominal_cols:
    le = LabelEncoder()
    le = le.fit(train[feature])
    train[feature] = le.transform(train[feature])

    unseen = [label for label in np.unique(test[feature]) if label not in le.classes_]
    if unseen:
        le.classes_ = np.append(le.classes_, unseen)
    test[feature] = le.transform(test[feature])

x_train = train.drop(['ID', 'stress_score'], axis=1)
y_train = train['stress_score']
x_test = test.drop('ID', axis=1)

print('x_train:', x_train.shape, '/ x_test:', x_test.shape)

x_train: (2994, 35) / x_test: (3000, 35)


## 5. 하이퍼파라미터 랜덤 탐색

`regularized`(#03) 근처 범위에서 20개 조합을 랜덤 샘플링해서 비교합니다.

In [6]:
# regularized(#03) 근처를 랜덤하게 세밀 탐색
import random
random.seed(RANDOM_STATE)

N_TRIALS = 20

search_space = {
    'n_estimators': [500, 600, 700, 800, 900, 1000, 1100, 1200],
    'learning_rate': [0.03, 0.04, 0.05, 0.06, 0.07, 0.08],
    'reg_alpha': [0.05, 0.1, 0.15, 0.2, 0.3],
    'reg_lambda': [0.05, 0.1, 0.15, 0.2, 0.3],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
}

trial_params = []
for _ in range(N_TRIALS):
    params = {k: random.choice(v) for k, v in search_space.items()}
    params['random_state'] = RANDOM_STATE
    trial_params.append(params)

# #03 기준값도 비교군으로 포함
trial_params.insert(0, dict(random_state=RANDOM_STATE, n_estimators=800, learning_rate=0.05,
                             reg_alpha=0.1, reg_lambda=0.1, subsample=0.8, colsample_bytree=0.8))

print(f'{len(trial_params)}개 조합 탐색 시작 (#03 기준값 포함)')

21개 조합 탐색 시작 (#03 기준값 포함)


In [9]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

search_results = []
for i, params in enumerate(trial_params):
    maes = []
    for tr_idx, val_idx in kf.split(x_train):
        X_tr, X_val = x_train.iloc[tr_idx], x_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        model = LGBMRegressor(**params, verbose=-1)
        model.fit(X_tr, y_tr)
        pred = model.predict(X_val)
        maes.append(mean_absolute_error(y_val, pred))
    mean_mae = np.mean(maes)
    search_results.append((mean_mae, params))
    tag = ' (#03 기준값)' if i == 0 else ''
    print(f'[{i+1}/{len(trial_params)}] MAE = {mean_mae:.4f}{tag}')

search_results.sort(key=lambda x: x[0])
best_mae, best_params = search_results[0]

print()
print(f'=== 최적 조합 (MAE {best_mae:.4f}) ===')
print(best_params)
print()

[1/21] MAE = 0.1842 (#03 기준값)
[2/21] MAE = 0.1994
[3/21] MAE = 0.1858
[4/21] MAE = 0.2024
[5/21] MAE = 0.1876
[6/21] MAE = 0.1801
[7/21] MAE = 0.1822
[8/21] MAE = 0.1990
[9/21] MAE = 0.1909
[10/21] MAE = 0.1848
[11/21] MAE = 0.1817
[12/21] MAE = 0.1905
[13/21] MAE = 0.1796
[14/21] MAE = 0.1804
[15/21] MAE = 0.1838
[16/21] MAE = 0.1841
[17/21] MAE = 0.1964
[18/21] MAE = 0.1937
[19/21] MAE = 0.1821
[20/21] MAE = 0.1847
[21/21] MAE = 0.1826

=== 최적 조합 (MAE 0.1796) ===
{'n_estimators': 1200, 'learning_rate': 0.08, 'reg_alpha': 0.15, 'reg_lambda': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'random_state': 42}



## 6. 최적 조합으로 최종 학습 + 제출 파일 저장

In [8]:
# 최적 조합으로 전체 데이터 학습 후 제출 파일 저장
import os

sample_submission = pd.read_csv('../data/sample_submission.csv')

final_model = LGBMRegressor(**best_params, verbose=-1)
final_model.fit(x_train, y_train)
pred = final_model.predict(x_test)

os.makedirs('../submissions', exist_ok=True)
sample_submission['stress_score'] = pred
sample_submission.to_csv('../submissions/submit_04_hyperparameter_finetuning.csv', index=False)
sample_submission.head()

,ID,stress_score
0,TEST_0000,0.601373
1,TEST_0001,0.924539
2,TEST_0002,0.192161
3,TEST_0003,0.501207
4,TEST_0004,0.576061
